Load schema definitions and config

In [0]:
%run ../config/config

In [0]:
gold_table = f"{catalog}.{gold_schema}.dim_date"

Create date dimension table for the year 2022 for analytics and aggregation

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DateType
from datetime import datetime
from datetime import timedelta

start_date = datetime(2022,1,1)
end_date = datetime(2022,12,30)

date_list = [(start_date + timedelta(days = x)).strftime('%Y-%m-%d') for x in range((end_date-start_date).days+1)]

gold_dim_date_df = spark.createDataFrame([(d,) for d in date_list], ['date'])
gold_dim_date_df= gold_dim_date_df.withColumn('date', F.col('date').cast(DateType()))

gold_dim_date_df = gold_dim_date_df.select(F.col('date'), F.year(F.col('date')).alias('year'), 
                           F.quarter(F.col('date')).alias('quarter'), 
                           F.month(F.col('date')).alias('month'), 
                           F.date_format(F.col('date'), 'MMMM').alias('month_name'), 
                           F.weekofyear(F.col('date')).alias('week_of_year'),
                           F.dayofyear(F.col('date')).alias('day_of_year'), 
                           F.dayofmonth(F.col('date')).alias('day'), 
                           F.dayofweek(F.col('date')).alias('day_of_week'), 
                           F.date_format(F.col('date'),'EEEE').alias('day_name'), 
                           F.when(F.dayofweek(F.col('date')).isin(1,7), 'Yes').otherwise('No').alias('Is_Weekend'))

gold_dim_date_df = gold_dim_date_df.withColumn("date_key",F.date_format("date", "yyyyMMdd").cast("int"))

gold_dim_date_df= gold_dim_date_df.select(
     "date_key",
     "year",
     "quarter",
     "month",
     "month_name",
     "week_of_year",
     "day",
     "day_name",
     "day_of_week",
     "day_of_year",
     F.col("Is_weekend").alias("weekend")
 )
                    

Write DataFrame to gold dim_date Delta table

In [0]:
gold_dim_date_df_write=(
    gold_dim_date_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(gold_table)
)